In [ ]:
# API-key setup — DO NOT hard-code your key in this cell.

import os


# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")


# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1")
MODEL = "llama-3.1-8b-instant"                # or your provider's model name

print("Client ready.")


In [ ]:
# TODO: Write a helper function you will reuse for the WHOLE lab:

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    print("Token usage:", response.usage)

    return response.choices[0].message.content


# Call it once with a simple question
answer = ask_llm("What is the capital of France?")
# TODO: Print response.usage as well — how many tokens did your call consume?
print(answer)

1. What is the difference between the system and user roles? Give an example of something that belongs in each.
The system role is
 the first instruction to the language model, and which defines the task or
role for the LLM, and sets overall tone and context for the LLM while the user role is the prompt or task given by the human user.

Eg.
System Role:
Claude is able to explain difficult concepts or ideas clearly.
It can also illustrate its explanations with examples, thought
experiments, or metaphors.

User Role:
Explain photosythesis


2. What is a token, roughly? Why do API providers bill per token rather than per request?

A token is an atomic unit an AI model reads.

API providers bill per token rather than per request because of the computational operation cost. With a large prompt more computal power would be used than a short one.
If it was done per request then it won't be fair to the different users who may be prompting differently with varying tokens used per their output.

In [ ]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
# A good test question: "Suggest a name for a savings product for market traders in Accra."

print("Temperature 0.0")
for i in range(5):
    answer = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=0.0)
    print(answer)

print("\nTemperature 1.2")
for i in range(5):
    answer = ask_llm("Suggest a name for a savings product for market traders in Accra.", temperature=1.2)
    print(answer)


# TODO: Print all 10 answers, grouped by temperature.


1. What did you observe at each temperature? For the loan decision-support system you are about to build, which temperature regime is appropriate, and why?

2. When the temperature was 0.0 the LLM gave the same answers everytime and correctly gave factual suggestions for the question
In the case of when the temperature was 1.2, the answers kept varying everytime and it eventually strayed of topic. It gave very creative answers although some didn't answer the question.

In [ ]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


In [ ]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

SUMMARY_PROMPT_V1 = """
Summarize this loan application:

{letter_text}
"""

prompt = SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L002"])
v1_l002 = ask_llm(prompt)


prompt = SUMMARY_PROMPT_V1.format(letter_text=LETTERS["L006"])
v1_l006 = ask_llm(prompt)




# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

SUMMARY_PROMPT_V2 = """
Summarize this loan application:

{letter_text}

"""

system_prompt = """
Summarize the loan application in a factual and neutral manner.
Do not invent, assume, or infer information that is not stated in the letter.
Include only information supported by the application.
Keep the summary concise, using 3-4 sentences.

"""

prompt2 = SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L002"])
v2_l002 = ask_llm(prompt2,system_prompt,temperature=0)

prompt2 = SUMMARY_PROMPT_V2.format(letter_text=LETTERS["L006"])
v2_l006 = ask_llm(prompt2,system_prompt,temperature=0)



# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

print("L002")
print("V1:", v1_l002)
print("V2:", v2_l002)

print("\nL006")
print("V1:", v1_l006)
print("V2:", v2_l006)



1. What concrete problems did V1's output have that V2 fixed? Quote examples.

V2 was straight to the point and did not add unnecessary detail for example:
In V1
Reason for urgency: Business has been slow, but expects a pickup after the festive season

V1 also presented information such as Trustworthiness for Kofi, even though this was only the applicant's own claim. V2 handled this more carefully by capturing it as:
Trustworthiness: Self-assessed by the applicant.

Lastly, V1 did not explicitly tell the model not to invent information. V2 added this constraint, which helped prevent unsupported assumptions, such as believing that an applicant has a good credit history or collateral simply because they seem trustworthy. It also helps prevent the assumption that businesses will succeed or that repayment is guaranteed, as shown in the report for the payment plan for Kofi.

2. Why is "no invented details" an essential instruction in this application? What is this failure mode called in the LLM literature?            

In a loan application, invented information can directly affect a financial decision. If an LLM makes up a good credit history, successful business experience, reliable income, collateral, or a realistic repayment ability, a loan officer could incorrectly judge the applicant as lower-risk.
The failure mode is called hallucination in LLM literature.
